# XRayVision AI — YOLOv8 Fracture Detection Training
**FYP: Web-Based AI Chatbot for Automated X-Ray Fracture Detection**  
Minhaj University Lahore — BSSE 8th Semester

---

## Before running: pick a dataset source

### Option A — Roboflow (Recommended)
1. Sign up free at [roboflow.com](https://roboflow.com)
2. Go to [universe.roboflow.com](https://universe.roboflow.com)
3. Search **bone fracture detection** → open any dataset
4. Click **Download** → choose **YOLOv8 format** → download zip
5. Upload the zip to Google Drive
6. **Run cells: 1 → 2 → 3 → 4A → 5 → 6 → 7 → 8**

### Option B — FracAtlas (No signup needed)
1. Go to [huggingface.co/datasets/FracAtlas/FracAtlas](https://huggingface.co/datasets/FracAtlas/FracAtlas)
2. Click **Files and versions** → download the zip
3. Upload zip to Google Drive
4. **Run cells: 1 → 2 → 3 → 4B → 5 → 6 → 7 → 8**

---
**First: Runtime → Change runtime type → T4 GPU → Save**

In [ ]:
# ── Cell 1: Verify GPU ────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available : {torch.cuda.is_available()}')
print(f'GPU            : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — go to Runtime > Change runtime type > T4"}')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────────
!pip install ultralytics -q
from ultralytics import YOLO
print('ultralytics OK')

In [ ]:
# ── Cell 3: Mount Google Drive ────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted')

In [ ]:
# ── Cell 4A: Extract Roboflow dataset (Option A) ──────────────────
# !! Skip this cell if using FracAtlas (Option B) !!
import os, zipfile

# Change this to your zip file path in Google Drive
ZIP_PATH    = '/content/drive/MyDrive/bone-fracture-detection.zip'
EXTRACT_DIR = '/content/dataset'

os.makedirs(EXTRACT_DIR, exist_ok=True)
print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)
print('Done!')

# Show top-level structure
for entry in sorted(os.listdir(EXTRACT_DIR)):
    print(f'  {entry}')

In [ ]:
# ── Cell 4B: Extract & convert FracAtlas (Option B) ───────────────
# !! Skip this cell if using Roboflow (Option A) !!
import os, json, shutil, zipfile
from pathlib import Path
from PIL import Image

ZIP_PATH    = '/content/drive/MyDrive/FracAtlas.zip'
EXTRACT_DIR = '/content/fracatlas_raw'
YOLO_DIR    = '/content/dataset'

# 1. Extract
print('Extracting...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(EXTRACT_DIR)

# 2. Locate images and COCO annotation files
def find_file(base, name):
    for p in Path(base).rglob(name):
        return str(p)
    return None

images_root  = find_file(EXTRACT_DIR, 'images') or EXTRACT_DIR
train_json   = find_file(EXTRACT_DIR, 'train.json')
test_json    = find_file(EXTRACT_DIR, 'test.json')  or find_file(EXTRACT_DIR, 'val.json')

print(f'Images root : {images_root}')
print(f'Train JSON  : {train_json}')
print(f'Val JSON    : {test_json}')

# 3. COCO → YOLO converter
def coco_to_yolo(coco_json_path, src_images_dir, out_images_dir, out_labels_dir):
    os.makedirs(out_images_dir, exist_ok=True)
    os.makedirs(out_labels_dir, exist_ok=True)

    with open(coco_json_path) as f:
        coco = json.load(f)

    id2info = {img['id']: img for img in coco['images']}
    # group annotations by image_id
    img_anns = {}
    for ann in coco.get('annotations', []):
        img_anns.setdefault(ann['image_id'], []).append(ann)

    for img_id, img_info in id2info.items():
        fname   = img_info['file_name']
        W, H    = img_info['width'], img_info['height']
        src_img = os.path.join(src_images_dir, os.path.basename(fname))

        if not os.path.exists(src_img):
            # search recursively
            found = find_file(EXTRACT_DIR, os.path.basename(fname))
            if found:
                src_img = found
            else:
                continue

        stem = Path(fname).stem
        shutil.copy(src_img, os.path.join(out_images_dir, os.path.basename(fname)))

        label_lines = []
        for ann in img_anns.get(img_id, []):
            cat_id = ann['category_id'] - 1   # 0-indexed
            x, y, w, h = ann['bbox']          # COCO: top-left x,y + w,h in pixels
            xc = (x + w / 2) / W
            yc = (y + h / 2) / H
            wn = w / W
            hn = h / H
            label_lines.append(f'{cat_id} {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}')

        label_path = os.path.join(out_labels_dir, stem + '.txt')
        with open(label_path, 'w') as lf:
            lf.write('\n'.join(label_lines))

    print(f'  Converted {len(id2info)} images → {out_images_dir}')

print('\nConverting train split...')
coco_to_yolo(train_json,
             images_root,
             f'{YOLO_DIR}/images/train',
             f'{YOLO_DIR}/labels/train')

print('Converting val split...')
coco_to_yolo(test_json,
             images_root,
             f'{YOLO_DIR}/images/valid',
             f'{YOLO_DIR}/labels/valid')

print('\nFracAtlas → YOLO conversion complete!')

In [ ]:
# ── Cell 5: Build dataset.yaml ────────────────────────────────────
import os, yaml

DATASET_DIR = '/content/dataset'

# Find images root
def find_images_root(base):
    for root, dirs, _ in os.walk(base):
        if 'images' in dirs:
            return root
    return base

dataset_root = find_images_root(DATASET_DIR)
images_dir   = os.path.join(dataset_root, 'images')
val_name     = 'valid' if os.path.exists(os.path.join(images_dir, 'valid')) else 'val'
has_train    = os.path.exists(os.path.join(images_dir, 'train'))
has_val      = os.path.exists(os.path.join(images_dir, val_name))

# Count images
def count_imgs(path):
    if not os.path.exists(path): return 0
    return sum(1 for f in os.listdir(path) if f.lower().endswith(('.jpg','.jpeg','.png')))

print(f'Dataset root : {dataset_root}')
print(f'Train images : {count_imgs(os.path.join(images_dir, "train"))}')
print(f'Val images   : {count_imgs(os.path.join(images_dir, val_name))}')

# Auto-detect class names from data.yaml if present (Roboflow includes one)
roboflow_yaml = os.path.join(dataset_root, 'data.yaml')
if os.path.exists(roboflow_yaml):
    with open(roboflow_yaml) as f:
        rf = yaml.safe_load(f)
    CLASSES = rf.get('names', ['fracture'])
    print(f'Classes from Roboflow yaml: {CLASSES}')
else:
    # FracAtlas classes
    CLASSES = ['Elbow Positive', 'Fingers Positive', 'Forearm Fracture',
               'Humerus Fracture', 'Shoulder Fracture', 'Wrist Positive']
    print(f'Using FracAtlas classes: {CLASSES}')

cfg = {
    'path'  : dataset_root,
    'train' : 'images/train' if has_train else 'images',
    'val'   : f'images/{val_name}' if has_val else 'images/train',
    'nc'    : len(CLASSES),
    'names' : CLASSES,
}

YAML_PATH = '/content/fracture_dataset.yaml'
with open(YAML_PATH, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'\ndataset.yaml saved to {YAML_PATH}')
!cat /content/fracture_dataset.yaml

In [ ]:
# ── Cell 6: TRAIN ─────────────────────────────────────────────────
# T4 GPU: ~1 hour for 50 epochs, ~2 hours for 100 epochs
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # swap to yolov8s.pt for +2% mAP, +30min

results = model.train(
    data      = '/content/fracture_dataset.yaml',
    epochs    = 50,
    batch     = 32,
    imgsz     = 640,
    project   = '/content/drive/MyDrive/xrayvision_training',
    name      = 'fracture_v1',
    device    = 0,
    # Medical X-ray augmentations
    hsv_h     = 0.0,    # X-rays are grayscale
    hsv_s     = 0.0,
    hsv_v     = 0.4,    # brightness variation (exposure)
    degrees   = 10.0,   # slight rotation
    fliplr    = 0.5,
    mosaic    = 0.5,
    patience  = 15,     # early stopping
    save_period = 10,
    plots     = True,
    verbose   = True,
    exist_ok  = True,
)

In [ ]:
# ── Cell 7: Validate ──────────────────────────────────────────────
from ultralytics import YOLO

BEST = '/content/drive/MyDrive/xrayvision_training/fracture_v1/weights/best.pt'
model   = YOLO(BEST)
metrics = model.val(data='/content/fracture_dataset.yaml')

print('\n========= FINAL RESULTS =========')
print(f'mAP50     : {metrics.box.map50:.4f}')
print(f'mAP50-95  : {metrics.box.map:.4f}')
print(f'Precision : {metrics.box.mp:.4f}')
print(f'Recall    : {metrics.box.mr:.4f}')
print('\nPer-class:')
for i, name in enumerate(metrics.names.values()):
    ap = metrics.box.ap50[i] if i < len(metrics.box.ap50) else 0
    print(f'  {name:<30} AP50={ap:.4f}')

In [ ]:
# ── Cell 8: Download best.pt ──────────────────────────────────────
BEST = '/content/drive/MyDrive/xrayvision_training/fracture_v1/weights/best.pt'

from google.colab import files
files.download(BEST)

print()
print('After downloading:')
print('  1. Copy best.pt  →  backend/models/fracture_yolov8.pt')
print('  2. Edit backend/.env:')
print('       YOLO_WEIGHTS_PATH=models/fracture_yolov8.pt')
print('       ALLOW_GENERIC_YOLO_WEIGHTS=false')
print('  3. Restart the FastAPI server')